# Part 1: The Core Metrics – The RAG Triad (Deep Dive)

When evaluating traditional machine learning models (like classification or regression), you have mathematical ground truths (e.g., accuracy, F1-score, mean squared error). In RAG systems, inputs and outputs are unstructured natural language. You cannot rely on exact string matching because two sentences can mean the exact same thing while using completely different words.

To solve this, modern evaluation architectures use LLM-as-a-Judge, treating a powerful model (like GPT-4o) as an impartial grader. The absolute gold standard framework for this is The RAG Triad.


## 1. The Three Variables of the RAG Triad

Every RAG evaluation pipeline monitors three core runtime artifacts:

User Query ($Q$): What the user asked.

Retrieved Context ($C$): The text chunks pulled from your vector store or graph database.

Generated Answer ($A$): The final response produced by your LLM.

The RAG Triad evaluates the relationships between these three variables across three distinct metrics:

### Metric 1: Context Relevance (Retriever Performance)

Relationship: Evaluated between User Query ($Q$) and Retrieved Context ($C$).

What it measures: Does the retrieved context contain only the information needed to answer the question, or is it bloated with irrelevant noise?

How LLM-as-a-Judge evaluates it:

The judge model inspects each retrieved chunk and asks: "What sentences in this chunk directly contribute to answering the user's query?

"Score Calculation: $\frac{\text{Number of relevant sentences in context}}{\text{Total number of sentences in context}}$. A low score means your vector search or hybrid retriever is pulling garbage data, bloating your prompt token window.

### Metric 2: Groundedness / Faithfulness (Hallucination Detection)

Relationship: Evaluated between Retrieved Context ($C$) and Generated Answer ($A$).

What it measures: Is every factual claim made in the generated answer strictly traceable back to the retrieved context? This is your primary defense against hallucinations.

How LLM-as-a-Judge evaluates it:

Step 1: The judge extracts all individual factual statements from the generated answer $A$.

Step 2: It checks each statement against the retrieved context $C$. If a claim cannot be verified using only the context, it is flagged as an unsupported hallucination.Score Calculation: $\frac{\text{Number of supported claims}}{\text{Total claims made in answer}}$. A score of $1.0$ means absolute faithfulness to the source data.

### Metric 3: Answer Relevance (End-to-End Quality)

Relationship: Evaluated between User Query ($Q$) and Generated Answer ($A$).

What it measures: Does the answer directly address what the user asked, or did the LLM deflect, ramble, or answer a completely different question? (Note: An answer can be 100% faithful to the context, but completely irrelevant to the user's prompt).

How LLM-as-a-Judge evaluates it:
The judge model looks at the final answer and reverse-engineers what question it appears to be answering. It then compares that implied question to the original user query $Q$.

Score Calculation: Scored on a continuous scale (e.g., $0$ to $1$) based on semantic alignment and completeness.

## Summary visual mapping


In [ ]:
[ User Query (Q) ] ------------ (Answer Relevance) -------------> [ Generated Answer (A) ]
               ^                                                                    ^
               |                                                                    |
     (Context Relevance)                                                        (Groundedness)
               |                                                                    |
               v                                                                    v
       [ Retrieved Context (C) ] ----------------------------------------------------